In [56]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import (
    accuracy_score, recall_score, precision_score, f1_score,
    roc_auc_score, log_loss, confusion_matrix, ConfusionMatrixDisplay,
    RocCurveDisplay, roc_curve
)
import mlflow
import mlflow.sklearn
import matplotlib.pyplot as plt

### 3. Importando o conjunto de dados ###

In [57]:
credito = pd.read_csv('Credit.csv')
credito.shape

(1000, 21)

### 4. Tratamento e Limpeza do Conjunto de dados ###

In [58]:
credito.head()

,checking_status,duration,credit_history,purpose,credit_amount,savings_status,employment,installment_commitment,personal_status,other_parties,...,property_magnitude,age,other_payment_plans,housing,existing_credits,job,num_dependents,own_telephone,foreign_worker,class
0,<0,6,'critical/other existing credit',radio/tv,1169,'no known savings',>=7,4,'male single',none,...,'real estate',67,none,own,2,skilled,1,yes,yes,good
1,0<=X<200,48,'existing paid',radio/tv,5951,<100,1<=X<4,2,'female div/dep/mar',none,...,'real estate',22,none,own,1,skilled,1,none,yes,bad
2,'no checking',12,'critical/other existing credit',education,2096,<100,4<=X<7,2,'male single',none,...,'real estate',49,none,own,1,'unskilled resident',2,none,yes,good
3,<0,42,'existing paid',furniture/equipment,7882,<100,4<=X<7,2,'male single',guarantor,...,'life insurance',45,none,'for free',1,skilled,2,none,yes,good
4,<0,24,'delayed previously','new car',4870,<100,1<=X<4,3,'male single',none,...,'no known property',53,none,'for free',2,skilled,2,none,yes,bad


In [59]:
#Os próximos scripts abaixo, transforma as colunas que estão type object em type categoria e separa o conjunto de treino e teste.
for col in credito.columns:
    if credito[col].dtype == 'object':
        credito[col] = credito[col].astype('category').cat.codes

In [60]:
previsores = credito.iloc[:,0:20].values
classe = credito.iloc[:,20].values

In [61]:
X_treinamento, X_teste, y_treinamento, y_teste = train_test_split(
    previsores,
    classe,
    test_size=0.3,
    random_state=123
)

TypeError: only integer scalar arrays can be converted to a scalar index

### 5. Pipeline de Treinamento do Algoritmo junto com validação e registrando resultados no MFLOW ###

In [ ]:
# A função abaixo é um conjunto de treinamento do algoritmo utilizando um pipeline automatizado para treinar diversos algoritmos e registrar os
# resultados no MLFLOW. Para isso, foi utilizada a biblioteca do Sklearn e, para calibragem dos algoritmos, o GrindSearch com cross validation

def treina_modelo(modelo, parametros):
    mlflow.set_experiment("Atividade_Modular")
    
    pipeline = Pipeline([
        ('clf', modelo)
    ])

    grid = GridSearchCV(pipeline, param_grid=parametros, cv=5, scoring='accuracy')

    with mlflow.start_run():
        grid.fit(X_treinamento, y_treinamento)
        
        best_model = grid.best_estimator_
        previsoes_train = best_model.predict(X_treinamento)
        previsoes_teste = best_model.predict(X_teste)
        
        # Registro de hiperparâmetros e parâmetros selecionados
        mlflow.log_params(grid.best_params_)

        # Métricas --- Treino
        acuracia_train = accuracy_score(y_treinamento, previsoes_train)
        recall_train = recall_score(y_treinamento, previsoes_train)
        precision_train = precision_score(y_treinamento, previsoes_train)
        f1_train = f1_score(y_treinamento, previsoes_train)
        auc_train = roc_auc_score(y_treinamento, previsoes_train)
        log_train = log_loss(y_treinamento, previsoes_train)

        # Registrar métricas
        mlflow.log_metric("acuracia_train", acuracia_train)
        mlflow.log_metric("recall_train", recall_train)
        mlflow.log_metric("precision_train", precision_train)
        mlflow.log_metric("f1_train", f1_train)
        mlflow.log_metric("auc_train", auc_train)
        mlflow.log_metric("log_train", log_train)    

        # Matriz de Confusão
        cm = confusion_matrix(y_treinamento, previsoes_train)
        confusion_disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        confusion_disp.plot()
        plt.savefig("confusion_matrix_train.png")
        mlflow.log_artifact("confusion_matrix_train.png")

        # Curva ROC
        if hasattr(best_model, "predict_proba"):  # Somente se o modelo tiver o método `predict_proba`
            fpr, tpr, _ = roc_curve(y_treinamento, best_model.predict_proba(X_treinamento)[:, 1])
            roc_disp = RocCurveDisplay(fpr=fpr, tpr=tpr)
            roc_disp.plot()
            plt.savefig("roc_curve_train.png")
            mlflow.log_artifact("roc_curve_train.png")
            
         # Métricas --- Teste
        acuracia_test = accuracy_score(y_teste, previsoes_teste)
        recall_test = recall_score(y_teste, previsoes_teste)
        precision_test = precision_score(y_teste, previsoes_teste)
        f1_test = f1_score(y_teste, previsoes_teste)
        auc_test = roc_auc_score(y_teste, previsoes_teste)
        log_test = log_loss(y_teste, previsoes_teste)

        # Registrar métricas
        mlflow.log_metric("acuracia_test", acuracia_test)
        mlflow.log_metric("recall_test", recall_test)
        mlflow.log_metric("precision_test", precision_test)
        mlflow.log_metric("f1_test", f1_test)
        mlflow.log_metric("auc_test", auc_test)
        mlflow.log_metric("log_test", log_test)    

        # Matriz de Confusão
        cm = confusion_matrix(y_teste, previsoes_teste)
        confusion_disp = ConfusionMatrixDisplay(confusion_matrix=cm)
        confusion_disp.plot()
        plt.savefig("confusion_matrix_test.png")
        mlflow.log_artifact("confusion_matrix_test.png")

        # Curva ROC
        if hasattr(best_model, "predict_proba"):  # Somente se o modelo tiver o método `predict_proba`
            fpr, tpr, _ = roc_curve(y_teste, best_model.predict_proba(X_teste)[:, 1])
            roc_disp = RocCurveDisplay(fpr=fpr, tpr=tpr)
            roc_disp.plot()
            plt.savefig("roc_curve_test.png")
            mlflow.log_artifact("roc_curve_test.png")

        # Logar o modelo
        mlflow.sklearn.log_model(best_model, "Modelo")

        print("Modelo treinado: ", mlflow.active_run().info.run_uuid)

    mlflow.end_run()

In [ ]:
# Configuração de modelos e hiperparâmetros
modelos_e_parametros = {
    RandomForestClassifier(): {
        'clf__n_estimators': [50, 100, 500]
    },
    LogisticRegression(max_iter=500): {
        'clf__C': [0.1, 1, 10],
        'clf__penalty': ['l2']
    },
    KNeighborsClassifier(): {
        'clf__n_neighbors': [3, 5, 7],
        'clf__weights': ['uniform', 'distance']
    }
}

In [ ]:
# Loop para treinar e registrar cada modelo com diferentes parâmetros
for modelo, parametros in modelos_e_parametros.items():
    print("Modelo:" , modelo)
    print("Parametros:" , parametros)
    treina_modelo(modelo, parametros)

Modelo: RandomForestClassifier()
Parametros: {'clf__n_estimators': [50, 100, 500]}


PermissionError: [WinError 5] Acesso negado: 'c:\\Users\\Henrique%E2%80%A0'